# Week 5 — Faster Sampling with DDIM

Last week I got a DDPM training and sampling on MNIST, but sampling took 1000 forward passes through the UNet to generate a single batch of images. That's painfully slow if you want to iterate on anything.

This week is about DDIM: same trained model, same training objective, but a different (deterministic) sampling equation that lets me skip most of the 1000 steps. I'm rebuilding the DDPM pieces from Week 4 here so this notebook is self-contained, then layering DDIM sampling on top of it without retraining anything.

In [ ]:
import math
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms
from torchvision.utils import make_grid
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
torch.manual_seed(0)

## Rebuilding the noise schedule

Same cosine scheduler as Week 3/4. The only thing I add here is a helper to pull out an arbitrary subsequence of timesteps, since DDIM doesn't need to walk through every single one of the 1000 steps.

In [ ]:
class NoiseScheduler:
    def __init__(self, timesteps=1000, s=0.008, device=device):
        self.timesteps = timesteps
        self.device = device

        steps = torch.arange(timesteps + 1, dtype=torch.float64) / timesteps
        f_t = torch.cos((steps + s) / (1 + s) * math.pi / 2) ** 2
        alphas_cumprod = f_t / f_t[0]
        alphas_cumprod = torch.clamp(alphas_cumprod, min=1e-9)

        self.alphas_cumprod = alphas_cumprod[1:].float().to(device)
        alphas_cumprod_prev = torch.cat([torch.tensor([1.0]), self.alphas_cumprod[:-1]])
        self.alphas_cumprod_prev = alphas_cumprod_prev.to(device)
        self.betas = (1 - self.alphas_cumprod / self.alphas_cumprod_prev).clamp(max=0.999)
        self.alphas = 1.0 - self.betas

    def add_noise(self, x0, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x0)
        sqrt_ac = self.alphas_cumprod[t].sqrt().view(-1, 1, 1, 1)
        sqrt_one_minus_ac = (1 - self.alphas_cumprod[t]).sqrt().view(-1, 1, 1, 1)
        return sqrt_ac * x0 + sqrt_one_minus_ac * noise, noise

scheduler = NoiseScheduler(timesteps=1000, device=device)

## Rebuilding the UNet

Identical architecture to Week 4 — base channels 64, sinusoidal timestep embeddings projected through a small MLP and added into every ResBlock. I'm not changing the architecture this week; the whole point is that DDIM reuses a model trained with the standard DDPM objective unchanged.

In [ ]:
class SinusoidalTimestepEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device).float() / half)
        args = t.float().unsqueeze(1) * freqs.unsqueeze(0)
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)


class TimestepMLP(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.embed = SinusoidalTimestepEmbedding(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, dim * 4), nn.SiLU(), nn.Linear(dim * 4, dim))

    def forward(self, t):
        return self.mlp(self.embed(t))


class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim):
        super().__init__()
        self.norm1 = nn.GroupNorm(min(8, in_ch), in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.time_proj = nn.Linear(time_dim, out_ch)
        self.norm2 = nn.GroupNorm(min(8, out_ch), out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, t_emb):
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.time_proj(t_emb)[:, :, None, None]
        h = self.conv2(F.silu(self.norm2(h)))
        return h + self.skip(x)


class Down(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim):
        super().__init__()
        self.block = ResBlock(in_ch, out_ch, time_dim)
        self.pool = nn.Conv2d(out_ch, out_ch, 3, stride=2, padding=1)

    def forward(self, x, t_emb):
        h = self.block(x, t_emb)
        return self.pool(h), h


class Up(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, time_dim):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, in_ch, 2, stride=2)
        self.block = ResBlock(in_ch + skip_ch, out_ch, time_dim)

    def forward(self, x, skip, t_emb):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.pad(x, (0, skip.shape[-1] - x.shape[-1], 0, skip.shape[-2] - x.shape[-2]))
        x = torch.cat([x, skip], dim=1)
        return self.block(x, t_emb)


class UNetDDPM(nn.Module):
    def __init__(self, in_ch=1, base_ch=64, time_dim=128):
        super().__init__()
        self.time_mlp = TimestepMLP(time_dim)
        self.inc = ResBlock(in_ch, base_ch, time_dim)
        self.down1 = Down(base_ch, base_ch * 2, time_dim)
        self.down2 = Down(base_ch * 2, base_ch * 4, time_dim)
        self.down3 = Down(base_ch * 4, base_ch * 8, time_dim)
        self.bottleneck = ResBlock(base_ch * 8, base_ch * 8, time_dim)
        self.up1 = Up(base_ch * 8, base_ch * 8, base_ch * 4, time_dim)
        self.up2 = Up(base_ch * 4, base_ch * 4, base_ch * 2, time_dim)
        # Down1's skip tensor has base_ch*2 channels (the ResBlock's out_ch, not in_ch) — skip_ch must match that.
        self.up3 = Up(base_ch * 2, base_ch * 2, base_ch, time_dim)
        self.outc = nn.Conv2d(base_ch, in_ch, 1)

    def forward(self, x, t):
        t_emb = self.time_mlp(t)
        h0 = self.inc(x, t_emb)
        h1, skip1 = self.down1(h0, t_emb)
        h2, skip2 = self.down2(h1, t_emb)
        h3, skip3 = self.down3(h2, t_emb)
        h3 = self.bottleneck(h3, t_emb)
        h = self.up1(h3, skip3, t_emb)
        h = self.up2(h, skip2, t_emb)
        h = self.up3(h, skip1, t_emb)
        return self.outc(h)

model = UNetDDPM().to(device)
print(sum(p.numel() for p in model.parameters()), "parameters")

## Data and a quick training pass

I don't need a brand new training run for this week conceptually — the whole point of DDIM is that it works on a model trained the normal DDPM way. But to actually have something to sample from in this notebook, I train for a modest number of epochs here (fewer than Week 4's full run, since the focus this week is the sampler, not chasing perfect sample quality).

In [ ]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
train_dataset = torchvision.datasets.MNIST(root="./data", train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)
epochs = 20
losses = []

for epoch in range(epochs):
    running_loss = 0.0
    for x0, _ in train_loader:
        x0 = x0.to(device)
        t = torch.randint(0, scheduler.timesteps, (x0.shape[0],), device=device)
        xt, noise = scheduler.add_noise(x0, t)
        pred_noise = model(xt, t)
        loss = F.mse_loss(pred_noise, noise)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        running_loss += loss.item() * x0.shape[0]

    epoch_loss = running_loss / len(train_dataset)
    losses.append(epoch_loss)
    print(f"epoch {epoch+1}/{epochs}  loss={epoch_loss:.4f}")

plt.plot(losses)
plt.xlabel("epoch")
plt.ylabel("MSE loss")
plt.title("Training loss")
plt.show()

## The DDIM update equation

DDPM's reverse step is stochastic: it adds a fresh noise sample at every step, which is why you need ~1000 small steps to keep the process well-behaved.

DDIM rewrites the reverse step as:

$$x_{t-1} = \sqrt{\bar\alpha_{t-1}} \left( \frac{x_t - \sqrt{1-\bar\alpha_t}\,\epsilon_\theta(x_t, t)}{\sqrt{\bar\alpha_t}} \right) + \sqrt{1-\bar\alpha_{t-1} - \sigma_t^2}\,\epsilon_\theta(x_t, t) + \sigma_t z$$

where the first parenthesized term is the model's estimate of $x_0$, and $\sigma_t$ is controlled by `eta`:

$$\sigma_t = \eta \sqrt{\frac{(1-\bar\alpha_{t-1})}{(1-\bar\alpha_t)}\left(1 - \frac{\bar\alpha_t}{\bar\alpha_{t-1}}\right)}$$

With `eta=0` there's no added noise at all — the whole trajectory from pure noise to image is deterministic given the starting noise. With `eta=1` this becomes equivalent to DDPM's ancestral sampling. That's a useful sanity check: if I plug eta=1 and use every single one of the 1000 timesteps, I should get DDPM samples back out.

The other piece is the timestep subset. Instead of stepping through t=999, 998, 997, ..., 0, I pick `num_steps` evenly spaced timesteps out of the original 1000 and only evaluate the model at those.

In [ ]:
@torch.no_grad()
def ddim_sample(model, scheduler, shape, num_steps=50, eta=0.0, device=device, x_T=None):
    model.eval()
    T = scheduler.timesteps

    # evenly spaced subset of timesteps, e.g. 999, 979, ..., 19, going down to 0
    step_indices = torch.linspace(0, T - 1, num_steps).long().flip(0).to(device)

    x = torch.randn(shape, device=device) if x_T is None else x_T

    ac = scheduler.alphas_cumprod

    for i in range(len(step_indices)):
        t = step_indices[i]
        t_batch = torch.full((shape[0],), t.item(), device=device, dtype=torch.long)

        ac_t = ac[t]
        ac_prev = ac[step_indices[i + 1]] if i + 1 < len(step_indices) else torch.tensor(1.0, device=device)

        eps = model(x, t_batch)

        x0_pred = (x - (1 - ac_t).sqrt() * eps) / ac_t.sqrt()
        x0_pred = x0_pred.clamp(-1, 1)

        sigma_t = eta * torch.sqrt((1 - ac_prev) / (1 - ac_t) * (1 - ac_t / ac_prev))
        dir_coeff = torch.sqrt((1 - ac_prev - sigma_t ** 2).clamp(min=0))

        noise = torch.randn_like(x) if (eta > 0 and t > 0) else torch.zeros_like(x)
        x = ac_prev.sqrt() * x0_pred + dir_coeff * eps + sigma_t * noise

    model.train()
    return x.clamp(-1, 1)

For the speed/quality comparison I also need the plain DDPM sampler from Week 4 (full 1000 steps, stochastic).

In [ ]:
@torch.no_grad()
def ddpm_sample(model, scheduler, shape, device=device):
    model.eval()
    x = torch.randn(shape, device=device)
    T = scheduler.timesteps

    for t in reversed(range(T)):
        t_batch = torch.full((shape[0],), t, device=device, dtype=torch.long)
        eps = model(x, t_batch)

        alpha_t = scheduler.alphas[t]
        ac_t = scheduler.alphas_cumprod[t]
        beta_t = scheduler.betas[t]

        mean = (1 / alpha_t.sqrt()) * (x - (beta_t / (1 - ac_t).sqrt()) * eps)

        if t > 0:
            noise = torch.randn_like(x)
            ac_prev = scheduler.alphas_cumprod_prev[t]
            var = beta_t * (1 - ac_prev) / (1 - ac_t)
            x = mean + var.sqrt() * noise
        else:
            x = mean

    model.train()
    return x.clamp(-1, 1)

## DDPM vs DDIM: speed and quality

Now the actual comparison: full DDPM (1000 steps) against DDIM at a handful of step counts. I'm timing wall-clock seconds for a fixed batch size on whatever device this notebook is running on, and eyeballing sample quality side by side.

In [ ]:
batch_shape = (16, 1, 28, 28)

results = {}

start = time.time()
ddpm_samples = ddpm_sample(model, scheduler, batch_shape)
results["DDPM (1000 steps)"] = (ddpm_samples, time.time() - start)

for n_steps in [250, 50, 20]:
    start = time.time()
    ddim_samples = ddim_sample(model, scheduler, batch_shape, num_steps=n_steps, eta=0.0)
    results[f"DDIM ({n_steps} steps, eta=0)"] = (ddim_samples, time.time() - start)

fig, axes = plt.subplots(1, len(results), figsize=(5 * len(results), 5))
for ax, (label, (samples, elapsed)) in zip(axes, results.items()):
    grid = make_grid(samples, nrow=4, normalize=True, value_range=(-1, 1))
    ax.imshow(grid.permute(1, 2, 0).cpu(), cmap="gray")
    ax.set_title(f"{label}\n{elapsed:.2f}s")
    ax.axis("off")
plt.tight_layout()
plt.show()

for label, (_, elapsed) in results.items():
    print(f"{label:30s} {elapsed:.3f}s")

## Sweeping eta

Holding the number of steps fixed at 50, I sweep eta from 0 (fully deterministic) up to 1 (DDPM-equivalent noise level) to see how the samples change. I'd expect eta=0 to produce the sharpest, most repeatable digits, with quality degrading slightly and variability increasing as eta rises, since fewer steps means each injected noise term has more relative impact.

In [ ]:
etas = [0.0, 0.25, 0.5, 1.0]
torch.manual_seed(42)
x_T = torch.randn(batch_shape, device=device)

fig, axes = plt.subplots(1, len(etas), figsize=(5 * len(etas), 5))
for ax, eta in zip(axes, etas):
    torch.manual_seed(42)
    samples = ddim_sample(model, scheduler, batch_shape, num_steps=50, eta=eta, x_T=x_T.clone())
    grid = make_grid(samples, nrow=4, normalize=True, value_range=(-1, 1))
    ax.imshow(grid.permute(1, 2, 0).cpu(), cmap="gray")
    ax.set_title(f"eta = {eta}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## Latent interpolation

Because DDIM is deterministic, the starting noise `x_T` behaves like a genuine latent code — the same `x_T` always maps to the same image. That means I can linearly interpolate between two noise vectors and decode each interpolated point with DDIM to get a smooth morph between two generated digits. This isn't really possible with vanilla DDPM since the stochastic noise injected at every step would make each generation independent even from the same starting point.

In [ ]:
torch.manual_seed(7)
x_T_a = torch.randn((1, 1, 28, 28), device=device)
x_T_b = torch.randn((1, 1, 28, 28), device=device)

alphas_interp = torch.linspace(0, 1, 8)
interp_samples = []
for a in alphas_interp:
    x_T_interp = (1 - a) * x_T_a + a * x_T_b
    sample = ddim_sample(model, scheduler, (1, 1, 28, 28), num_steps=50, eta=0.0, x_T=x_T_interp)
    interp_samples.append(sample)

interp_grid = make_grid(torch.cat(interp_samples, dim=0), nrow=8, normalize=True, value_range=(-1, 1))
plt.figure(figsize=(16, 3))
plt.imshow(interp_grid.permute(1, 2, 0).cpu(), cmap="gray")
plt.axis("off")
plt.title("Latent interpolation between two noise vectors")
plt.show()

## Image-to-image via partial noising

I can also start DDIM from a partially-noised real image instead of pure noise. Take a real digit, push it forward to some intermediate timestep `t_start` using the forward process, then run the DDIM reverse process only from `t_start` down to 0. The lower `t_start` is, the closer the output stays to the original image; the higher it is, the more the model is free to change it.

In [ ]:
@torch.no_grad()
def ddim_img2img(model, scheduler, x0, t_start, num_steps=50, eta=0.0, device=device):
    model.eval()
    T = scheduler.timesteps

    full_steps = torch.linspace(0, T - 1, num_steps).long()
    step_indices = full_steps[full_steps <= t_start].flip(0).to(device)
    if len(step_indices) == 0:
        return x0

    t_tensor = torch.tensor([step_indices[0]], device=device)
    xt, _ = scheduler.add_noise(x0, t_tensor)
    x = xt
    ac = scheduler.alphas_cumprod

    for i in range(len(step_indices)):
        t = step_indices[i]
        t_batch = torch.full((x.shape[0],), t.item(), device=device, dtype=torch.long)

        ac_t = ac[t]
        ac_prev = ac[step_indices[i + 1]] if i + 1 < len(step_indices) else torch.tensor(1.0, device=device)

        eps = model(x, t_batch)
        x0_pred = ((x - (1 - ac_t).sqrt() * eps) / ac_t.sqrt()).clamp(-1, 1)

        sigma_t = eta * torch.sqrt((1 - ac_prev) / (1 - ac_t) * (1 - ac_t / ac_prev))
        dir_coeff = torch.sqrt((1 - ac_prev - sigma_t ** 2).clamp(min=0))
        noise = torch.randn_like(x) if (eta > 0 and t > 0) else torch.zeros_like(x)
        x = ac_prev.sqrt() * x0_pred + dir_coeff * eps + sigma_t * noise

    model.train()
    return x.clamp(-1, 1)

real_img, _ = train_dataset[0]
real_img = real_img.unsqueeze(0).to(device)

fig, axes = plt.subplots(1, 5, figsize=(15, 4))
axes[0].imshow(real_img[0, 0].cpu(), cmap="gray")
axes[0].set_title("original")
axes[0].axis("off")

for ax, t_start in zip(axes[1:], [100, 300, 500, 800]):
    out = ddim_img2img(model, scheduler, real_img, t_start=t_start, num_steps=50)
    ax.imshow(out[0, 0].cpu(), cmap="gray")
    ax.set_title(f"t_start={t_start}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## Self-check questions

**1. How does DDIM use the same trained model as DDPM but sample faster?**

The training objective never required the reverse process to be Markovian or stochastic — the model is only ever trained to predict the noise in $x_t$ given $x_t$ and $t$, for whatever $t$ happens to get sampled during training. DDIM exploits this by defining a different, non-Markovian reverse process that shares the exact same marginal distributions $q(x_t | x_0)$ as DDPM but doesn't need to visit every intermediate $t$. Since the network was only ever asked to denoise at a given $t$ in isolation, you can call it at a sparser set of timesteps and the deterministic update equation still holds — no retraining, no architecture change, just a different sampling-time equation built on the same $\epsilon_\theta$.

**2. What does the `eta` parameter control? What does `eta=0` mean? `eta=1`?**

`eta` scales how much fresh stochastic noise gets injected at each reverse step, via the $\sigma_t$ term in the update equation. At `eta=0` the update is fully deterministic — given the same starting $x_T$, you always get the same output, and the whole trajectory traces a fixed path through the data manifold (this is what makes latent interpolation and img2img possible below). At `eta=1`, with every timestep used (no skipping), $\sigma_t$ recovers exactly the posterior variance $q(x_{t-1} | x_t, x_0)$ that DDPM's reverse step uses, so the update becomes numerically identical to DDPM's ancestral sampling, noise injection and all.

**3. How does DDIM choose its smaller timestep subset from the original schedule?**

It picks an evenly spaced subsequence of the original 1000 timesteps — e.g. for 50 steps, roughly every 20th timestep (999, 979, 959, ..., 19, with appropriate rounding). In my `ddim_sample` implementation above this is just `torch.linspace(0, T-1, num_steps)`. The schedule still has to start near $T$ (pure noise) and end at 0 (clean image); skipping unevenly or non-monotonically would break the assumption that each step's $\bar\alpha$ values bracket a sensible chunk of the noise schedule.

**4. What's the trade-off: when would you prefer DDPM over DDIM?**

DDIM wins on speed almost unconditionally for a model already trained the standard DDPM way, so for anything interactive — a demo, an img2img tool, fast iteration during development — DDIM at `eta=0` is the obvious choice. The case for DDPM is when you specifically want the stochasticity: DDPM's ancestral sampling gives you a different sample every time from the same starting noise, which is useful if "maximum diversity per training run" matters more than speed, or if you're trying to characterize the model's full output distribution rather than get one good sample fast. In practice, for this notebook's MNIST model the DDIM samples at 50 steps look essentially as good as the full 1000-step DDPM samples, so there's no real quality reason to prefer DDPM here — only the diversity argument would push me back toward it.